# Inference comparison: SmallUNet and MediumUNet

This Colab notebook loads two trained checkpoints from Google Drive:

- `SmallUNet.pt`
- `MediumUNet.pth`

It then runs inference on `train_020333_rgb.png` and plots the same three-panel visualization used in the training notebooks:

1. RGB image
2. Ground-truth depth
3. Predicted depth

The notebook expects the corresponding ground-truth file to be named `train_020333_depth.npy`.


In [ ]:
# ============================================================
# 1. Mount Google Drive and configure paths
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# TODO: change this to the Drive folder containing your checkpoints.
# Example: Path('/content/drive/MyDrive/monodepth_experiments')
DRIVE_DIR = Path('/content/drive/MyDrive')

SMALL_CKPT_PATH = DRIVE_DIR / 'SmallUNet.pt'
MEDIUM_CKPT_PATH = DRIVE_DIR / 'MediumUNet.pth'

# If your train images are in another folder, set DATA_ROOT directly.
# If left as None, the notebook searches DRIVE_DIR and common KaggleHub locations.
DATA_ROOT = None

SAMPLE_RGB_NAME = 'train_020333_rgb.png'

OUTPUT_DIR = DRIVE_DIR / 'inference_visualizations'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Drive folder:', DRIVE_DIR)
print('Small checkpoint:', SMALL_CKPT_PATH)
print('Medium checkpoint:', MEDIUM_CKPT_PATH)
print('Output folder:', OUTPUT_DIR)


In [ ]:
# ============================================================
# 2. Imports and device
# ============================================================

import os
import re
import math
import warnings
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# 3. Model definitions
# ============================================================
# These definitions reproduce the architectures used in your notebooks.
# The loader below tries several compatible candidates because the filenames
# and active notebook classes were not always perfectly aligned.

class TinyDoubleConv(nn.Module):
    """Double-convolution block used by the baseline TinyUNet/SmallUNet notebook."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class TinyUNet(nn.Module):
    """Baseline model from SmallUNet notebook; the notebook instantiated TinyUNet()."""
    def __init__(self):
        super().__init__()
        self.enc1 = TinyDoubleConv(3, 16)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = TinyDoubleConv(16, 32)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = TinyDoubleConv(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = TinyDoubleConv(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = TinyDoubleConv(32, 16)
        self.out_conv = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.up2(b)
        if d2.shape[2:] != e2.shape[2:]:
            d2 = F.interpolate(d2, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return torch.sigmoid(self.out_conv(d1))


class ConvBlock(nn.Module):
    """GroupNorm + SiLU block used by the stronger U-Net variants."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        groups = max(1, min(8, out_ch // 4))
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class DoubleConv(nn.Module):
    """GroupNorm + SiLU double-convolution block."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        groups = max(1, min(16, out_ch // 4))
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class MediumUNet(nn.Module):
    """Medium U-Net architecture from MediumUNet notebook."""
    def __init__(self, in_channels=3, out_channels=1, features=(32, 64, 128)):
        super().__init__()
        features = list(features)
        self.encoder1 = DoubleConv(in_channels, features[0])
        self.pool1 = nn.MaxPool2d(2)
        self.encoder2 = DoubleConv(features[0], features[1])
        self.pool2 = nn.MaxPool2d(2)
        self.encoder3 = DoubleConv(features[1], features[2])
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(features[2], features[2] * 2)
        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(features[2] * 2, features[2], kernel_size=3, padding=1),
        )
        self.decoder3 = DoubleConv(features[2] * 2, features[2])
        self.up2 = nn.ConvTranspose2d(features[2], features[1], kernel_size=2, stride=2)
        self.decoder2 = DoubleConv(features[1] * 2, features[1])
        self.up1 = nn.ConvTranspose2d(features[1], features[0], kernel_size=2, stride=2)
        self.decoder1 = DoubleConv(features[0] * 2, features[0])
        self.out_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        e1 = self.encoder1(x)
        e2 = self.encoder2(self.pool1(e1))
        e3 = self.encoder3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))

        d3 = self.up3(b)
        if d3.shape[2:] != e3.shape[2:]:
            d3 = F.interpolate(d3, size=e3.shape[2:], mode='bilinear', align_corners=False)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.decoder3(d3)

        d2 = self.up2(d3)
        if d2.shape[2:] != e2.shape[2:]:
            d2 = F.interpolate(d2, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.decoder2(d2)

        d1 = self.up1(d2)
        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.decoder1(d1)
        return torch.sigmoid(self.out_conv(d1))


class SmallUNet3Level(nn.Module):
    """Fallback SmallUNet definition present in the MediumUNet notebook."""
    def __init__(self):
        super().__init__()
        self.enc1 = ConvBlock(3, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(128, 256)
        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
        )
        self.dec3 = ConvBlock(256, 128)
        self.up2 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(128, 64)
        self.up1 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(64, 32)
        self.out_conv = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.up3(b)
        if d3.shape[2:] != e3.shape[2:]:
            d3 = F.interpolate(d3, size=e3.shape[2:], mode='bilinear', align_corners=False)
        d3 = torch.cat([d3, e3], dim=1)
        d3 = self.dec3(d3)
        d2 = self.up2(d3)
        if d2.shape[2:] != e2.shape[2:]:
            d2 = F.interpolate(d2, size=e2.shape[2:], mode='bilinear', align_corners=False)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)
        d1 = self.up1(d2)
        if d1.shape[2:] != e1.shape[2:]:
            d1 = F.interpolate(d1, size=e1.shape[2:], mode='bilinear', align_corners=False)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)
        return torch.sigmoid(self.out_conv(d1))


class ASPPBottleneck(nn.Module):
    """Atrous Spatial Pyramid Pooling bottleneck used by BetterUNet."""
    def __init__(self, in_ch, out_ch, rates=(1, 6, 12, 18)):
        super().__init__()
        mid = in_ch // len(rates)
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_ch, mid, 3, padding=r, dilation=r, bias=False),
                nn.GroupNorm(max(1, mid // 4), mid),
                nn.SiLU(inplace=True),
            ) for r in rates
        ])
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_ch, mid, 1, bias=False),
            nn.SiLU(inplace=True),
        )
        fused = mid * (len(rates) + 1)
        self.project = nn.Sequential(
            nn.Conv2d(fused, out_ch, 1, bias=False),
            nn.GroupNorm(max(1, out_ch // 4), out_ch),
            nn.SiLU(inplace=True),
            nn.Dropout2d(0.1),
        )

    def forward(self, x):
        h, w = x.shape[2:]
        parts = [b(x) for b in self.branches]
        pooled = F.interpolate(self.pool(x), size=(h, w), mode='bilinear', align_corners=False)
        return self.project(torch.cat(parts + [pooled], dim=1))


class BetterUNet(nn.Module):
    """4-level U-Net with GroupNorm, SiLU, and ASPP bottleneck."""
    def __init__(self, in_channels=3, out_channels=1, features=(64, 128, 256, 512)):
        super().__init__()
        self.features = tuple(features)
        f = self.features
        self.enc1 = DoubleConv(in_channels, f[0])
        self.enc2 = DoubleConv(f[0], f[1])
        self.enc3 = DoubleConv(f[1], f[2])
        self.enc4 = DoubleConv(f[2], f[3])
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ASPPBottleneck(f[3], f[3] * 2)
        self.up4 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), nn.Conv2d(f[3] * 2, f[3], 1, bias=False))
        self.dec4 = DoubleConv(f[3] * 2, f[3])
        self.up3 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), nn.Conv2d(f[3], f[2], 1, bias=False))
        self.dec3 = DoubleConv(f[2] * 2, f[2])
        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), nn.Conv2d(f[2], f[1], 1, bias=False))
        self.dec2 = DoubleConv(f[1] * 2, f[1])
        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False), nn.Conv2d(f[1], f[0], 1, bias=False))
        self.dec1 = DoubleConv(f[0] * 2, f[0])
        self.out_conv = nn.Conv2d(f[0], out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        def up_cat_conv(up, dec, feat, skip):
            d = up(feat)
            if d.shape[2:] != skip.shape[2:]:
                d = F.interpolate(d, size=skip.shape[2:], mode='bilinear', align_corners=False)
            return dec(torch.cat([d, skip], dim=1))

        d4 = up_cat_conv(self.up4, self.dec4, b, e4)
        d3 = up_cat_conv(self.up3, self.dec3, d4, e3)
        d2 = up_cat_conv(self.up2, self.dec2, d3, e2)
        d1 = up_cat_conv(self.up1, self.dec1, d2, e1)
        return torch.sigmoid(self.out_conv(d1))


class DepthCueExtractor(nn.Module):
    """Hand-crafted monocular cues used by DepthModelWithCues."""
    def __init__(self):
        super().__init__()
        sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1,-2,-1],[0,0,0],[1,2,1]], dtype=torch.float32)
        laplacian = torch.tensor([[0,1,0],[1,-4,1],[0,1,0]], dtype=torch.float32)
        self.register_buffer('sobel_x', sobel_x.view(1, 1, 3, 3))
        self.register_buffer('sobel_y', sobel_y.view(1, 1, 3, 3))
        self.register_buffer('laplacian', laplacian.view(1, 1, 3, 3))

    def forward(self, x):
        gray = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        gx = F.conv2d(gray, self.sobel_x, padding=1)
        gy = F.conv2d(gray, self.sobel_y, padding=1)
        grad_mag = torch.sqrt(gx ** 2 + gy ** 2)
        lap = F.conv2d(gray, self.laplacian, padding=1).abs()

        B, _, H, W = x.shape
        yy, xx = torch.meshgrid(
            torch.linspace(0, 1, H, device=x.device),
            torch.linspace(0, 1, W, device=x.device),
            indexing='ij',
        )
        xx = xx.expand(B, 1, H, W)
        yy = yy.expand(B, 1, H, W)
        blur = F.avg_pool2d(gray, kernel_size=15, stride=1, padding=7)
        shading = gray / (blur + 1e-3)
        return torch.cat([gray, grad_mag, lap, xx, yy, shading], dim=1)


class DepthModelWithCues(nn.Module):
    """Cue-augmented model from the Medium/DepthModelWithCues experiments."""
    def __init__(self, features=(32, 64, 128, 256)):
        super().__init__()
        self.features = tuple(features)
        self.cue_extractor = DepthCueExtractor()
        self.unet = BetterUNet(in_channels=9, features=features)

    def forward(self, x):
        cues = self.cue_extractor(x)
        cues = (cues - cues.mean(dim=(2, 3), keepdim=True)) / (cues.std(dim=(2, 3), keepdim=True) + 1e-6)
        cues = torch.clamp(cues, -5.0, 5.0)
        x_aug = torch.cat([x, cues], dim=1)
        return self.unet(x_aug)


In [ ]:

# ============================================================
# 4. Utilities: checkpoint loading, sample loading, inference, plotting
# ============================================================

def extract_state_dict(checkpoint_obj):
    """Return a PyTorch state_dict from either a raw state_dict or a checkpoint dictionary."""
    if isinstance(checkpoint_obj, dict):
        for key in ['model_state_dict', 'state_dict', 'model', 'net']:
            if key in checkpoint_obj and isinstance(checkpoint_obj[key], dict):
                return checkpoint_obj[key], checkpoint_obj
        # Raw state_dict case: all values are tensors/parameters.
        if checkpoint_obj and all(torch.is_tensor(v) or isinstance(v, torch.nn.Parameter) for v in checkpoint_obj.values()):
            return checkpoint_obj, {}
    raise ValueError('Could not find a model state_dict in the checkpoint.')


def strip_common_prefixes(state_dict):
    """Remove common wrappers such as module. from DataParallel checkpoints."""
    prefixes = ['module.', 'model.', 'net.']
    candidates = [state_dict]
    for prefix in prefixes:
        if any(k.startswith(prefix) for k in state_dict.keys()):
            candidates.append({k[len(prefix):] if k.startswith(prefix) else k: v for k, v in state_dict.items()})
    return candidates


def checkpoint_architecture_hint(checkpoint_meta):
    cfg = checkpoint_meta.get('model_config', {}) if isinstance(checkpoint_meta, dict) else {}
    for key in ['architecture', 'arch', 'model_name', 'name']:
        if key in cfg:
            return str(cfg[key])
        if isinstance(checkpoint_meta, dict) and key in checkpoint_meta:
            return str(checkpoint_meta[key])
    return None


def load_best_matching_model(checkpoint_path, candidates, device=DEVICE):
    """
    Try to load a checkpoint into a list of candidate architectures.

    candidates is a list of tuples:
        (display_name, constructor, img_size)
    """
    checkpoint_path = Path(checkpoint_path)
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

    # weights_only=False keeps compatibility with full checkpoint dictionaries in newer PyTorch versions.
    try:
        checkpoint_obj = torch.load(checkpoint_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint_obj = torch.load(checkpoint_path, map_location=device)
    state_dict, meta = extract_state_dict(checkpoint_obj)
    state_dict_candidates = strip_common_prefixes(state_dict)

    hint = checkpoint_architecture_hint(meta)
    if hint:
        print(f'Checkpoint architecture hint for {checkpoint_path.name}: {hint}')
        # Try matching hinted architecture first.
        candidates = sorted(candidates, key=lambda c: 0 if hint.lower() in c[0].lower() or c[0].lower() in hint.lower() else 1)

    errors = []
    for display_name, constructor, img_size in candidates:
        for sd in state_dict_candidates:
            model = constructor().to(device)
            try:
                model.load_state_dict(sd, strict=True)
                model.eval()
                n_params = sum(p.numel() for p in model.parameters())
                print(f'Loaded {checkpoint_path.name} as {display_name} ({n_params/1e6:.2f}M params, img_size={img_size})')
                return model, display_name, img_size, meta
            except RuntimeError as e:
                errors.append(f'[{display_name}] {str(e).splitlines()[0]}')

    message = '\n'.join(errors[:12])
    raise RuntimeError(
        f'Could not load checkpoint {checkpoint_path} into any candidate architecture.\n'
        f'First errors:\n{message}\n\n'
        f'If this happens, check that the architecture definition matches the training notebook exactly.'
    )


def find_sample_rgb(sample_name, data_root=None, drive_dir=None):
    """Find train_020333_rgb.png in a specific root, Drive folder, or common KaggleHub paths."""
    roots = []
    if data_root is not None:
        roots.append(Path(data_root))
    if drive_dir is not None:
        roots.append(Path(drive_dir))

    common_roots = [
        Path('/root/.cache/kagglehub/competitions/ethz-cil-monocular-depth-estimation-2026/monodepth_kaggle2026/train'),
        Path('/content/.cache/kagglehub/competitions/ethz-cil-monocular-depth-estimation-2026/monodepth_kaggle2026/train'),
        Path('/content/monodepth_kaggle2026/train'),
        Path('/content/train'),
    ]
    roots.extend(common_roots)

    checked = []
    for root in roots:
        if root is None or not root.exists():
            continue
        checked.append(str(root))
        direct = root / sample_name
        if direct.exists():
            return direct
        matches = list(root.rglob(sample_name))
        if matches:
            return matches[0]

    raise FileNotFoundError(
        f'Could not find {sample_name}. Searched these roots:\n' + '\n'.join(checked) +
        '\n\nSet DATA_ROOT to the folder containing the train images, or put the sample in DRIVE_DIR.'
    )


def load_rgb_depth_pair(rgb_path, img_size):
    """Load and resize RGB/depth exactly like the training notebooks."""
    rgb_path = Path(rgb_path)
    depth_path = Path(str(rgb_path).replace('_rgb.png', '_depth.npy'))
    if not depth_path.exists():
        raise FileNotFoundError(f'Ground-truth depth file not found: {depth_path}')

    rgb = np.array(Image.open(rgb_path).convert('RGB'), dtype=np.float32) / 255.0
    depth = np.load(depth_path).astype(np.float32)

    rgb_t = torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)
    rgb_t = F.interpolate(rgb_t, size=(img_size, img_size), mode='bilinear', align_corners=False)

    depth_t = torch.from_numpy(depth).unsqueeze(0).unsqueeze(0)
    depth_t = F.interpolate(depth_t, size=(img_size, img_size), mode='nearest')

    valid_mask = (depth_t > 0).float()
    depth_t = torch.clamp(depth_t, min=0.0, max=80.0) / 80.0

    return rgb_t, depth_t, valid_mask


def run_inference(model, rgb_path, img_size, device=DEVICE):
    rgb_t, depth_t, mask_t = load_rgb_depth_pair(rgb_path, img_size)
    image = rgb_t.to(device)
    model.eval()
    with torch.no_grad():
        pred = model(image)
        pred = torch.clamp(pred, min=0.0, max=1.0)
    return {
        'image': rgb_t[0].permute(1, 2, 0).cpu().numpy(),
        'gt': depth_t[0, 0].cpu().numpy(),
        'pred': pred[0, 0].detach().cpu().numpy(),
        'mask': mask_t[0, 0].cpu().numpy(),
    }


def masked_copy(arr, mask):
    out = arr.copy()
    out[mask == 0] = np.nan
    return out


def plot_triptych(result, title, sample_name, save_path=None, display_meters=False, shared_depth_scale=False):
    """Replicate the notebook's RGB / Ground Truth Depth / Predicted Depth plot."""
    img = result['image']
    gt = masked_copy(result['gt'], result['mask'])
    pred = masked_copy(result['pred'], result['mask'])

    if display_meters:
        gt = gt * 80.0
        pred = pred * 80.0
        depth_label = 'Depth (m, clipped to 80m)'
    else:
        depth_label = 'Normalized depth'

    vmin = vmax = None
    if shared_depth_scale:
        both = np.concatenate([gt[np.isfinite(gt)].ravel(), pred[np.isfinite(pred)].ravel()])
        if both.size:
            vmin = float(np.nanpercentile(both, 1))
            vmax = float(np.nanpercentile(both, 99))

    plt.figure(figsize=(14, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(img)
    plt.title('RGB')
    plt.axis('off')

    plt.subplot(1, 3, 2)
    im = plt.imshow(gt, cmap='viridis', vmin=vmin, vmax=vmax)
    plt.title('Ground Truth Depth')
    plt.axis('off')
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label(depth_label)

    plt.subplot(1, 3, 3)
    im = plt.imshow(pred, cmap='viridis', vmin=vmin, vmax=vmax)
    plt.title('Predicted Depth')
    plt.axis('off')
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label(depth_label)

    plt.suptitle(f'{title} — {sample_name}')
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print('Saved:', save_path)

    plt.show()


def plot_combined(results, sample_name, save_path=None, display_meters=False):
    """One compact comparison figure: RGB, GT, SmallUNet pred, MediumUNet pred."""
    first = next(iter(results.values()))
    img = first['image']
    gt = masked_copy(first['gt'], first['mask'])

    if display_meters:
        gt = gt * 80.0
        depth_label = 'Depth (m, clipped to 80m)'
    else:
        depth_label = 'Normalized depth'

    pred_arrays = {}
    all_depth = [gt[np.isfinite(gt)].ravel()]
    for name, result in results.items():
        pred = masked_copy(result['pred'], result['mask'])
        if display_meters:
            pred = pred * 80.0
        pred_arrays[name] = pred
        all_depth.append(pred[np.isfinite(pred)].ravel())

    both = np.concatenate([x for x in all_depth if x.size])
    vmin = float(np.nanpercentile(both, 1)) if both.size else None
    vmax = float(np.nanpercentile(both, 99)) if both.size else None

    ncols = 2 + len(results)
    plt.figure(figsize=(4 * ncols, 4))

    plt.subplot(1, ncols, 1)
    plt.imshow(img)
    plt.title('RGB')
    plt.axis('off')

    plt.subplot(1, ncols, 2)
    im = plt.imshow(gt, cmap='viridis', vmin=vmin, vmax=vmax)
    plt.title('Ground Truth')
    plt.axis('off')
    cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
    cbar.set_label(depth_label)

    for j, (name, pred) in enumerate(pred_arrays.items(), start=3):
        plt.subplot(1, ncols, j)
        im = plt.imshow(pred, cmap='viridis', vmin=vmin, vmax=vmax)
        plt.title(f'{name}\nPrediction')
        plt.axis('off')
        cbar = plt.colorbar(im, fraction=0.046, pad=0.04)
        cbar.set_label(depth_label)

    plt.suptitle(sample_name)
    plt.tight_layout()

    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print('Saved:', save_path)

    plt.show()


In [ ]:
# ============================================================
# 5. Find the sample and load both checkpoints
# ============================================================

rgb_path = find_sample_rgb(SAMPLE_RGB_NAME, data_root=DATA_ROOT, drive_dir=DRIVE_DIR)
print('Sample RGB:', rgb_path)
print('Sample depth:', Path(str(rgb_path).replace('_rgb.png', '_depth.npy')))

small_candidates = [
    ('TinyUNet / baseline SmallUNet notebook', TinyUNet, 128),
    ('SmallUNet3Level fallback', SmallUNet3Level, 256),
]

medium_candidates = [
    ('MediumUNet', MediumUNet, 256),
    ('DepthModelWithCues fallback', DepthModelWithCues, 256),
    ('BetterUNet fallback', BetterUNet, 320),
]

small_model, small_name, small_img_size, small_meta = load_best_matching_model(
    SMALL_CKPT_PATH,
    small_candidates,
    device=DEVICE,
)

medium_model, medium_name, medium_img_size, medium_meta = load_best_matching_model(
    MEDIUM_CKPT_PATH,
    medium_candidates,
    device=DEVICE,
)


In [ ]:
# ============================================================
# 6. Run inference and create the same triptych plot for SmallUNet
# ============================================================

small_result = run_inference(small_model, rgb_path, small_img_size, device=DEVICE)

plot_triptych(
    small_result,
    title=small_name,
    sample_name=SAMPLE_RGB_NAME,
    save_path=OUTPUT_DIR / 'SmallUNet_train_020333_triptych.png',
    display_meters=False,       # False matches your notebook: normalized depth in [0,1]
    shared_depth_scale=False,   # False matches your notebook: each colorbar autoscaled
)


In [ ]:
# ============================================================
# 7. Run inference and create the same triptych plot for MediumUNet
# ============================================================

medium_result = run_inference(medium_model, rgb_path, medium_img_size, device=DEVICE)

plot_triptych(
    medium_result,
    title=medium_name,
    sample_name=SAMPLE_RGB_NAME,
    save_path=OUTPUT_DIR / 'MediumUNet_train_020333_triptych.png',
    display_meters=False,       # False matches your notebook: normalized depth in [0,1]
    shared_depth_scale=False,   # False matches your notebook: each colorbar autoscaled
)


In [ ]:
# ============================================================
# 8. Optional: compact side-by-side comparison of both predictions
# ============================================================
# This plot uses a shared depth color scale, which makes comparison easier.

comparison_results = {
    'SmallUNet': small_result,
    'MediumUNet': medium_result,
}

plot_combined(
    comparison_results,
    sample_name=SAMPLE_RGB_NAME,
    save_path=OUTPUT_DIR / 'SmallUNet_vs_MediumUNet_train_020333.png',
    display_meters=False,
)


## Notes

- The plots above use normalized depth values, matching the notebook cell where prediction was shown without multiplying by 80.
- To display approximate metric depth instead, change `display_meters=False` to `display_meters=True` in the plotting calls.
- If checkpoint loading fails, the checkpoint was likely saved from a slightly different architecture definition. In that case, copy the exact model class from the training notebook into the model-definition cell above.
